<a href="https://colab.research.google.com/github/Anikrai11/My_All_Project/blob/main/Copy_of_Rag_analysis_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf  sentence-transformers faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 92.7 MB/s eta 0:00:00


In [ ]:
import os
from sentence_transformers import SentenceTransformer
import faiss
from pypdf import PdfReader
import numpy as np
from openai import OpenAI

In [ ]:

embed_model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
from transformers import pipeline
generator=pipeline("text-generation",
                   model="google/flan-t5-large",
                   device=0)

# step_1

In [ ]:
def load_pdf(pdf_path):
  reader=PdfReader(pdf_path)
  text=""
  for i,page in enumerate(reader.pages):
    text+=f"\n[page{i+1}]\n"+page.extract_text()
  return text

# step_2

In [ ]:
def split_text(text,chunk_size=500,overloop=50):
  words=text.split()
  chunks=[]
  for i in range(0,len(words),chunk_size-overloop):

   chunk=" ".join(words[i:i+chunk_size])
   chunks.append(chunk)
  return chunks

# step_3

In [ ]:
def create_index(chunks):
  embeddings=embed_model.encode(chunks,show_progress_bar=True)
  dimension=embeddings.shape[1]
  index=faiss.IndexFlatL2(dimension)
  index.add(np.array(embeddings))
  return index,embeddings


# retriver for question

#step_4

In [ ]:
def retrieve(query,chunks,index,k=3):
  query_vector=embed_model.encode([query])
  D,I=index.search(np.array(query_vector),k)
  result=[]
  for i in I[0]:
    result.append(chunks[i])
  return result


# step_5

In [ ]:
def generate_answer(query, context):
    prompt = f"""নিচের Context ব্যবহার করে প্রশ্নের উত্তর দাও।
যদি Context এ উত্তর না থাকে তাহলে বলো "আমি জানি না"।

Context:
{context}

প্রশ্ন: {query}
উত্তর:"""

    result = generator(prompt, max_length=300)
    return result[0]['generated_text']

# step_6

In [ ]:
if __name__== "__main__":
  pdf_path="/content/7 habits.pdf"
  print("1.pdf_loading")
  text=load_pdf(pdf_path)

  print("2.spliting")
  chunks=split_text(text)
  print(f"Total_chunks:{len(chunks)}")
  print("3.creating vactor index")
  index,embeddings=create_index(chunks)
  while True:
    query=input("\nyour question")
    if query=="exit":break
    print("searching")
    result=retrieve(query,chunks,index)
    context="\n---\n".join(result)
    print("genarating")
    answer=generate_answer(query,context)
    print(answer)
    print("\ndd",result[0][:300],"---")

